In [19]:
# libs
#import re
import pysam
import duckdb
import pandas as pd
import regex as re
import numpy as np
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict

In [22]:
# path to ref
ref_c1 = '../references/data_genomika_constrained_indexed.fasta'

# list of BAM files
bams = {
    "hp_c1": "../data/CAMELLIA_11/camellia_11_trim_filter_9_mapped.bam",
    "ns_c1": "../data/CAMELLIA_17/camellia_17_trim_filter_9_mapped.bam",
}

In [30]:
def rrs_align_bam(bam_dict, out=None):

    motif_re = re.compile(r"GGATG|CATCC")
    data = []

    for name, bam_file in bam_dict.items():
        bam = pysam.AlignmentFile(bam_file, "rb")
        
        for read in bam:
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue

            seq = read.query_sequence
            if not seq:
                continue

            # find motif matches in the read
            for m in motif_re.finditer(seq):
                mpos = m.start()
                motif_seq = m.group()

                # map read positions to reference positions
                ref_positions = read.get_reference_positions(full_length=True)
                if mpos < len(ref_positions):
                    motif_ref_pos = ref_positions[mpos]
                else:
                    motif_ref_pos = None

                row = {
                    "sample": name,
                    "read_name": read.query_name,
                    "ref_id": bam.get_reference_name(read.reference_id),
                    "read_start": read.reference_start,
                    "read_end": read.reference_end,
                    "motif": motif_seq,
                    "motif_read_pos": mpos,
                    "motif_ref_pos": motif_ref_pos,
                    "NM": read.get_tag("NM") if read.has_tag("NM") else None
                }
                data.append(row)
        
        bam.close()

    df = pd.DataFrame(data)

    if out and not df.empty:
        df.to_parquet(out, index=False)

    return df

In [31]:
# get primary alignments
res_rrs = rrs_align_bam(bams)

In [32]:
res_trunc = (
    res_rrs
    .query("ref_id in ['0093','0335']")
    .query("motif_ref_pos.notna()")
)
res_trunc

,sample,read_name,ref_id,read_start,read_end,motif,motif_read_pos,motif_ref_pos,NM
2171,hp_c1,4eaf568f-6913-4b5d-ab84-a3423560221d,0093,0,140,CATCC,126,56.0,9
2175,hp_c1,69a05174-c631-49e9-b9a8-d96386536980,0093,0,136,CATCC,94,75.0,11
2176,hp_c1,69a05174-c631-49e9-b9a8-d96386536980,0093,0,136,CATCC,124,104.0,11
2180,hp_c1,ade3981d-e619-4906-bde1-9566e0350fae,0093,0,140,CATCC,152,104.0,3
2183,hp_c1,824d1f91-50e2-494f-9f71-0006535ae8ff,0093,0,138,CATCC,151,104.0,2
2184,hp_c1,90b00c4d-102d-402b-bbcf-7feb4000d5fa,0093,0,140,CATCC,147,104.0,3
2185,hp_c1,6490c85e-fec9-44bb-a59b-360be78ef160,0093,0,140,CATCC,149,104.0,1
2186,hp_c1,c57df18c-9a76-43b5-9a28-e28471981cab,0093,0,140,CATCC,126,104.0,5
2187,hp_c1,0b9b0b93-f467-4841-b0b1-221caa19b3a3,0093,0,140,CATCC,120,104.0,8
2189,hp_c1,e1d1267b-e80b-4e10-a49a-4e559b28cb04,0093,0,140,CATCC,126,104.0,7
